In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [2]:
optimizer = torch.optim.AdamW(
    model.parameters(),   # All trainable weights
    lr = 5e-5
)

In [3]:
text = "Python is a programming language"
inputs = tokenizer(
    text, 
    return_tensors = "pt"
)
print(inputs)

{'input_ids': tensor([[30280,   374,   264, 15473,  4128]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


In [4]:
outputs = model(
    **inputs,
    labels = inputs["input_ids"]
)

In [5]:
outputs

CausalLMOutputWithPast(loss=tensor(1.6037, grad_fn=<NllLossBackward0>), logits=tensor([[[ 7.0625,  7.9375,  8.2500,  ..., -4.2812, -4.2812, -4.2812],
         [ 8.1875,  7.8750,  7.6875,  ..., -3.0781, -3.0781, -3.0781],
         [ 5.3438,  7.1562,  4.8750,  ..., -5.2500, -5.2500, -5.2500],
         [ 9.0000,  8.4375,  6.7188,  ..., -4.6875, -4.6875, -4.6875],
         [13.0000,  9.7500,  9.1250,  ..., -2.7656, -2.7500, -2.7656]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)

In [6]:
loss = outputs.loss
loss

tensor(1.6037, grad_fn=<NllLossBackward0>)

## Observe one Parameter

In [7]:
parameter = next(model.parameters())
print(parameter.view(-1)[0]) # Before training

tensor(-0.0101, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [8]:
loss.backward()

In [9]:
print(parameter.grad.view(-1)[0])  # This gardient tell optimzier how to adjust that weight

# Interpretation:
# Positive gradient → decrease the weight.
# Negative gradient → increase the weight.
# Zero gradient → no change.

tensor(-8.1539e-05, dtype=torch.bfloat16)


### Update the Weights

In [ ]:
optimizer.step()

# Optimizer updates every parameter using its gradient

# Conceptually:
# New Weight = Old Weight - Learning Rate × Gradient
# For example:
# Old Weight = 0.50
# Gradient = 0.20
# Learning Rate = 0.01
# New Weight = 0.50 - (0.01 × 0.20)
# New Weight = 0.498
# Millions of weights are updated simultaneously.

In [11]:
print(parameter.view(-1)[0]) # After training

tensor(-0.0100, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [ ]:
optimizer.zero_grad() 

# PyTorch accumulates gradients by default.
# Without clearing them, gradients from previous iterations would be added to new ones.
# A typical training loop always ends with

In [19]:
# Overall one training iteration looks like this:
outputs = model(
    **inputs,
    labels = inputs['input_ids']
)

loss = outputs.loss
loss.backward()
optimizer.step()
optimizer.zero_grad()